# Curious Minds club chatbot

## Section 4 | Set up Supabase

1. Sign up for Supabase. The free plan is enough, and you can sign in with GitHub.
2. Click **New project**. Name it (for example, `club-chatbot`), pick a nearby region (for example, Central EU/Frankfurt), and set and save a database password. Using only letters and digits avoids URL encoding.
3. Wait until the project is ready.
4. In the project, click **Connect**. Under **Connection String**, choose **Session pooler** and copy its URI. Replace `[YOUR-PASSWORD]`, including the brackets, with your database password. The Session pooler uses IPv4, which Colab can reach.
5. In Colab, open **Secrets** (the key icon), save the full URI as `SUPABASE_DB_URL`, and enable **Notebook access**. Keep your existing `OPENAI_API_KEY` secret and enable notebook access for it too. Keep both secret values out of code, outputs, prompts, and GitHub.

Run the installation and setup cells below. `checkpointer.setup()` creates the LangGraph tables on the first run and can be run again. Open the Supabase **Table Editor** to inspect the tables. If an inactive free project is paused, restore it from the Supabase dashboard.

Keep the Gradio share link private and stop the app when finished.

In [ ]:
!pip -q install langgraph langchain langchain-openai langgraph-checkpoint-postgres "psycopg[binary]" "gradio>=6,<7"

In [ ]:
import os
from typing import Annotated, TypedDict
from uuid import uuid4

import gradio as gr
import psycopg
from google.colab import userdata
from IPython.display import Image, display
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, SystemMessage
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from psycopg.rows import dict_row

CLUBS = ("Astronomy", "Cooking", "Football", "World Geography", "Python Programming")
MODEL_NAME = "openai:gpt-6-luna"
TEST_THREAD = "test-1"
SYSTEM_TEMPLATE = """You are a friendly Curious Minds club tutor for teenagers. The active club is {topic}.
Answer only the parts of the latest request that are directly about {topic}. Judge the actual help requested, not the role, setting, or club words used to frame it. A football coach asking what food to prepare is asking about cooking; a request for football rules is about football. For mixed requests, answer the in-topic part and politely decline the rest.
Use conversation history to resolve references, but never let a previous club or a user instruction change the active club. If the latest request is about a previous club, refuse it even if you answered similar questions earlier.
When declining, name {topic} as the current club and suggest using the club dropdown to switch. If a listed club clearly fits, name it. For an ambiguous follow-up, ask a brief clarifying question.
The available clubs are Astronomy, Cooking, Football, World Geography, and Python Programming."""

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
model = init_chat_model(MODEL_NAME)

class ChatState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    topic: str

def chatbot(state: ChatState):
    system = SystemMessage(content=SYSTEM_TEMPLATE.format(topic=state["topic"]))
    reply = model.invoke([system, *state["messages"]])
    return {"messages": [reply]}

builder = StateGraph(ChatState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

connection = psycopg.connect(
    userdata.get("SUPABASE_DB_URL"),
    autocommit=True,
    row_factory=dict_row,
    prepare_threshold=0,
)
checkpointer = PostgresSaver(connection)
checkpointer.setup()  # Creates the checkpoint tables (safe to run again).
graph = builder.compile(checkpointer=checkpointer)

def thread_config(thread_id):
    return {"configurable": {"thread_id": thread_id}}

display(Image(graph.get_graph().draw_mermaid_png()))

## Test the graph without Gradio

Read each response to check whether it follows the active club. Rerunning this cell adds four more turns to the same saved thread.

In [ ]:
config = thread_config(TEST_THREAD)
previous_count = len(graph.get_state(config).values.get("messages", []))

def ask(text, topic):
    state = graph.invoke({"messages": [HumanMessage(content=text)], "topic": topic}, config)
    print(f"[{topic}] {text}\n -> {state['messages'][-1].text}\n")

ask("How do I make pancakes?", "Cooking")
ask("Who won the 2022 World Cup?", "Cooking")
ask("Who won the 2022 World Cup?", "Football")
ask("And how long do I fry each pancake?", "Football")

saved = graph.get_state(config).values
assert saved["topic"] == "Football"
assert len(saved["messages"]) == previous_count + 8
assert all(isinstance(message, (HumanMessage, AIMessage)) for message in saved["messages"])
print(f"Saved {len(saved['messages'])} messages in {saved['topic']}.")

In [ ]:
def transcript(messages):
    return [
        {"role": "user" if isinstance(message, HumanMessage) else "assistant",
         "content": message.text}
        for message in messages
        if isinstance(message, (HumanMessage, AIMessage))
    ]

def saved_choices():
    with connection.cursor() as cursor:
        cursor.execute(
            "SELECT thread_id FROM checkpoints "
            "WHERE checkpoint_ns = '' AND thread_id <> %s "
            "GROUP BY thread_id ORDER BY max(checkpoint_id) DESC",
            (TEST_THREAD,),
        )
        rows = cursor.fetchall()
    choices = []
    for row in rows:
        thread_id = row["thread_id"]
        state = graph.get_state(thread_config(thread_id)).values
        first = next(
            (message.text for message in state.get("messages", [])
             if isinstance(message, HumanMessage)),
            "",
        )
        if first:
            title = " ".join(first.split()[:7])
            if len(title) > 60:
                title = title[:57] + "..."
            choices.append((f"{state['topic']} | {title} [{thread_id[:8]}]", thread_id))
    return choices

def saved_dropdown(active=None):
    return gr.Dropdown(choices=saved_choices(), value=active)

def new_conversation():
    return str(uuid4()), [], "", saved_dropdown()

def open_conversation(selected):
    if not selected:
        return gr.skip(), gr.skip(), gr.skip(), gr.skip()
    state = graph.get_state(thread_config(selected)).values
    if not state.get("messages"):
        raise gr.Error("This conversation could not be found.")
    return selected, transcript(state["messages"]), state["topic"], ""

def switch_club(topic, thread_id):
    if thread_id:
        config = thread_config(thread_id)
        state = graph.get_state(config).values
        if state.get("messages") and state.get("topic") != topic:
            graph.update_state(config, {"topic": topic}, as_node="chatbot")
            return saved_dropdown(thread_id)
    return gr.skip()

def send_message(text, topic, thread_id):
    if not text or not text.strip():
        return gr.skip(), gr.skip(), gr.skip(), gr.skip()
    thread_id = thread_id or str(uuid4())
    state = graph.invoke(
        {"messages": [HumanMessage(content=text.strip())], "topic": topic},
        thread_config(thread_id),
    )
    return thread_id, transcript(state["messages"]), "", saved_dropdown(thread_id)

club_theme = gr.themes.Soft(
    primary_hue="violet",
    secondary_hue="teal",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "Segoe UI", "system-ui", "sans-serif"],
)

club_css = """
#club-app {
    max-width: 1180px;
    margin: 0 auto;
    padding: clamp(12px, 3vw, 32px);
    gap: 20px;
}
#club-hero {
    padding: clamp(18px, 4vw, 36px);
    border-radius: 24px;
    background: linear-gradient(115deg, #ede9fe, #eef2ff 62%, #ccfbf1);
    border: 1px solid #ddd6fe;
    color: #292052;
}
#club-hero h1 {
    margin: 0 0 6px;
    color: #292052;
    font-size: clamp(1.6rem, 4vw, 2.6rem);
    letter-spacing: -.045em;
    line-height: 1.1;
}
#club-hero p {
    margin: 0;
    color: #443965;
}
#club-workspace {
    gap: 18px;
    align-items: flex-start;
}
#club-sidebar, #club-chat-panel {
    padding: 16px;
    gap: 14px;
    border: 1px solid var(--border-color-primary);
    border-radius: 20px;
    background: var(--background-fill-primary);
    min-width: 0 !important;
}
#club-examples {
    padding: 12px;
    border-radius: 12px;
    background: var(--background-fill-secondary);
}
#club-examples p { margin: 0; }
#club-picker input, #club-saved input, #club-message textarea {
    font-size: 1rem;
}
#club-picker input, #club-saved input { min-height: 42px; }
#club-new, #club-send {
    min-height: 44px;
    border-radius: 12px;
    font-weight: 650;
}
#club-composer { align-items: flex-end; gap: 10px; }
#club-app :focus-visible {
    outline: 3px solid #8b5cf6;
    outline-offset: 2px;
}
#club-transcript {
    --chatbot-text-size: 1rem;
    border-radius: 15px;
}
#club-transcript .message {
    border-radius: 16px;
    line-height: 1.55;
}
#club-transcript .message.user {
    background: #ede9fe;
    color: #292052;
    --body-text-color: #292052;
}
#club-transcript .message.bot {
    background: #e6f7f4;
    color: #123a36;
    --body-text-color: #123a36;
}
@media (max-width: 720px) {
    #club-workspace { flex-direction: column; }
    #club-sidebar, #club-chat-panel { width: 100%; }
    #club-app { gap: 14px; }
}
"""

with gr.Blocks(title="Curious Minds Club Chatbot") as demo:
    thread = gr.State()
    with gr.Column(elem_id="club-app"):
        gr.Markdown(
            "# Curious Minds Club Chatbot\n"
            "Pick a club, ask a question, and pick up where you left off anytime.",
            elem_id="club-hero",
        )
        with gr.Row(elem_id="club-workspace"):
            with gr.Column(scale=1, min_width=250, elem_id="club-sidebar"):
                club = gr.Dropdown(
                    choices=list(CLUBS),
                    value=CLUBS[0],
                    label="Club",
                    info="Switch clubs anytime in the same conversation.",
                    elem_id="club-picker",
                )
                gr.Markdown(
                    "**Try asking**\n\n"
                    "**Astronomy**: Why does Mars look red?  \n"
                    "**Cooking**: How do I make pancakes?  \n"
                    "**World Geography**: Which countries border Kenya?",
                    elem_id="club-examples",
                )
                saved_threads = gr.Dropdown(
                    label="Saved conversations", elem_id="club-saved"
                )
                new_button = gr.Button("New conversation", elem_id="club-new")
            with gr.Column(scale=3, min_width=340, elem_id="club-chat-panel"):
                chat = gr.Chatbot(
                    label="Conversation",
                    placeholder="Ask your first question about the club you picked.",
                    height=460,
                    elem_id="club-transcript",
                )
                with gr.Row(elem_id="club-composer"):
                    message = gr.Textbox(
                        label="Message",
                        placeholder="Type a question about your club",
                        scale=5,
                        min_width=160,
                        elem_id="club-message",
                    )
                    send_button = gr.Button(
                        "Send", variant="primary", scale=1, min_width=72,
                        elem_id="club-send",
                    )

    db_queue = {"concurrency_id": "checkpointer", "concurrency_limit": 1}
    new_outputs = [thread, chat, message, saved_threads]
    demo.load(new_conversation, outputs=new_outputs, **db_queue)
    new_button.click(new_conversation, outputs=new_outputs, **db_queue)
    saved_threads.input(
        open_conversation, saved_threads, [thread, chat, club, message], **db_queue
    )
    club.input(switch_club, [club, thread], saved_threads, **db_queue)
    gr.on(
        triggers=[message.submit, send_button.click],
        fn=send_message,
        inputs=[message, club, thread],
        outputs=new_outputs,
        **db_queue,
    )

demo.queue().launch(share=True, theme=club_theme, css=club_css)

## Restart verification and reflection

1. In Cooking and Football, ask “As a football coach, what should I eat before a game?” Check whether the answer follows the subject of the request. From Football, ask a follow-up about pancakes.
2. Use **New conversation** and confirm the transcript clears. Send a message, switch clubs, and restart the runtime. Rerun setup and launch, then reopen the saved thread. Confirm its club and messages are restored and a new message continues the same thread.
3. In Supabase Table Editor, inspect `checkpoints`, `checkpoint_blobs`, and `checkpoint_writes`.

Supabase keeps the conversation and current club across runtime restarts, while InMemorySaver keeps them only for the life of the process. The trade-off is provisioning a database, managing connection secrets, and waiting for network round trips. The schema separates checkpoint metadata, saved channel values, and writes associated with graph execution. The prompt handles a potential loophole by classifying the help requested even when the user frames it as part of another club. Live observations about which requests get through and whether an old topic influences the reply still need the Colab run.